# Olist E-commerce — EDA: Tableau Data Preparation

## 1. Load Data

In [18]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data/cleaned_data'
OUT_DIR  = '../data/tableau_data'
os.makedirs(OUT_DIR, exist_ok=True)

def save(df, name):
    path = os.path.join(OUT_DIR, name)
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(f'  Saved {name}: {df.shape[0]:,} rows x {df.shape[1]} cols')

master = pd.read_excel(os.path.join(DATA_DIR, 'olist_master_review.xlsx'))

# Parse dates — errors='coerce' handles non-date strings (e.g. 'Not Available')
master['order_purchase_timestamp']      = pd.to_datetime(master['order_purchase_timestamp'], errors='coerce')
master['order_delivered_customer_date'] = pd.to_datetime(master['order_delivered_customer_date'], errors='coerce')

print(f'Loaded master: {master.shape[0]:,} rows x {master.shape[1]} cols')
master.head(3)

Loaded master: 119,142 rows x 26 cols


,order_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,customer_zip_code_prefix,customer_city,customer_state,product_id,seller_id,...,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,lat_median_customer,lng_median_customer,lat_median_seller,lng_median_seller
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13,2017-09-20,28013,Campos Dos Goytacazes,RJ,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,...,1.0,credit_card,2.0,72.19,97ca439bc427b48bc1cd7177abe71365,5.0,-21.763186,-41.310265,-22.497188,-44.127324
1,7d19f4ef4d04461989632411b7e588b9,91a792fef70ecd8cc69d3c7feb3d12da,delivered,2017-08-10,2017-08-24,36400,Conselheiro Lafaiete,MG,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,...,1.0,credit_card,4.0,72.19,426f43a82185969503fb3c86241a9535,5.0,-20.655379,-43.776331,-22.497188,-44.127324
2,130898c0987d1801452a8ed92a670612,e6eecc5a77de221464d1c4eaff0a9b64,delivered,2017-06-28,2017-07-13,75800,Jatai,GO,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,...,1.0,boleto,1.0,73.86,b11cba360bbe71410c291b764753d37f,5.0,-17.884731,-51.717133,-22.497188,-44.127324


## 2. NaN Analysis

In [19]:
null_counts = master.isnull().sum()
null_pct    = (null_counts / len(master) * 100).round(2)
nan_df = pd.DataFrame({'NaN Count': null_counts, 'NaN %': null_pct})
nan_df = nan_df[nan_df['NaN Count'] > 0].sort_values('NaN Count', ascending=False)
print(f'Columns with NaN: {len(nan_df)}')
display(nan_df)

Columns with NaN: 19


,NaN Count,NaN %
order_delivered_customer_date,3420,2.87
product_category_name_english,2576,2.16
lat_median_seller,1098,0.92
lng_median_seller,1098,0.92
review_score,997,0.84
review_id,997,0.84
product_id,833,0.70
seller_id,833,0.70
price,833,0.70
freight_value,833,0.70


## 3. NaN Cleaning Pipeline

| Group | Columns | Strategy |
|-------|---------|----------|
| A | `order_delivered_customer_date`, `price`, `product_id`, ... | Filter `delivered` then Drop |
| B | `product_category_name_english` | Fill `'Unknown'` |
| C | `review_id`, `review_score` | Keep as-is (no review = valid) |
| D | `lat/lng_median_*` | Fill with median per state |
| E | `payment_value`, `payment_type` | Drop (only 6 rows) |

In [20]:
print(f'Initial rows: {len(master):,}')

# Group A: Keep only delivered orders
master = master[master['order_status'] == 'delivered'].copy()
print(f"After filter 'delivered':  {len(master):,}")

# Group A: Drop rows missing product info
master = master.dropna(subset=['price', 'product_id']).copy()
print(f'After drop missing product: {len(master):,}')

# Group E: Drop 6 rows missing payment info
master = master.dropna(subset=['payment_value', 'payment_type']).copy()
print(f'After drop missing payment: {len(master):,}')

# Group B: Fill unknown category
master['product_category_name_english'] = master['product_category_name_english'].fillna('Unknown')

# Group D: Fill lat/lng with median per state
for col, grp in [
    ('lat_median_customer', 'customer_state'),
    ('lng_median_customer', 'customer_state'),
    ('lat_median_seller',   'seller_state'),
    ('lng_median_seller',   'seller_state'),
]:
    master[col] = master.groupby(grp)[col].transform(lambda x: x.fillna(x.median()))

# Computed fields
master['revenue']        = (master['price'] + master['freight_value']).round(2)
master['delivery_days']  = (master['order_delivered_customer_date'] - master['order_purchase_timestamp']).dt.days
master['order_year']     = master['order_purchase_timestamp'].dt.year
master['order_month']    = master['order_purchase_timestamp'].dt.month
master['order_quarter']  = master['order_purchase_timestamp'].dt.quarter
master['order_ym']       = master['order_purchase_timestamp'].dt.to_period('M').astype(str)
master['order_date']     = master['order_purchase_timestamp'].dt.date.astype(str)
master['order_dow_name'] = master['order_purchase_timestamp'].dt.day_name()

master['delivery_bucket'] = pd.cut(
    master['delivery_days'], bins=[-1, 7, 14, 21, 30, 999],
    labels=['<=7 days', '8-14 days', '15-21 days', '22-30 days', '>30 days']
)
master['review_label'] = master['review_score'].map(
    lambda s: 'No Review' if pd.isna(s) else
              ('Positive(4-5)' if s >= 4 else ('Neutral(3)' if s == 3 else 'Negative(1-2)'))
)

print(f'\nClean dataset ready: {len(master):,} rows')

Initial rows: 119,142
After filter 'delivered':  115,722
After drop missing product: 115,722
After drop missing payment: 115,719

Clean dataset ready: 115,719 rows


## 4. Export Tableau Data

In [21]:
# --- fact_orders_enriched: main fact table, one row per order item ---
save(master, 'fact_orders_enriched.csv')

  Saved fact_orders_enriched.csv: 115,719 rows x 36 cols


In [22]:
# --- agg_monthly_kpi: monthly KPI rollup ---
monthly = master.groupby('order_ym').agg(
    total_revenue     = ('revenue', 'sum'),
    total_payment     = ('payment_value', 'sum'),
    order_count       = ('order_id', 'nunique'),
    unique_customers  = ('customer_id', 'nunique'),
    avg_delivery_days = ('delivery_days', 'mean'),
    avg_review_score  = ('review_score', 'mean'),
).reset_index()

monthly['aov']              = (monthly['total_payment'] / monthly['order_count']).round(2)
monthly['avg_delivery_days']= monthly['avg_delivery_days'].round(1)
monthly['avg_review_score'] = monthly['avg_review_score'].round(2)
monthly['total_revenue']    = monthly['total_revenue'].round(2)

monthly = monthly.sort_values('order_ym')
monthly['revenue_prev']   = monthly['total_revenue'].shift(1)
monthly['mom_growth_pct'] = np.where(
    monthly['revenue_prev'] > 0,
    ((monthly['total_revenue'] - monthly['revenue_prev']) / monthly['revenue_prev'] * 100).round(2),
    np.nan
)
monthly.drop(columns=['revenue_prev'], inplace=True)
save(monthly, 'agg_monthly_kpi.csv')
display(monthly.tail(5))

  Saved agg_monthly_kpi.csv: 22 rows x 9 cols


,order_ym,total_revenue,total_payment,order_count,unique_customers,avg_delivery_days,avg_review_score,aov,mom_growth_pct
17,2018-04,1174143.49,1469136.33,6798,6798,11.3,4.12,216.11,0.10
18,2018-05,1171614.43,1481529.96,6749,6749,11.3,4.16,219.52,-0.22
19,2018-06,1064996.68,1286923.47,6099,6099,9.1,4.21,211.01,-9.10
20,2018-07,1066593.79,1309951.88,6159,6159,8.8,4.27,212.69,0.15
21,2018-08,1020340.91,1211344.79,6351,6351,7.6,4.25,190.73,-4.34


In [23]:
# --- agg_category_performance: revenue by product category ---
cat_perf = master.groupby(['product_category_name_english', 'order_year', 'order_quarter']).agg(
    total_revenue    = ('revenue', 'sum'),
    order_count      = ('order_id', 'nunique'),
    unique_customers = ('customer_id', 'nunique'),
    avg_price        = ('price', 'mean'),
    avg_review       = ('review_score', 'mean'),
    avg_delivery     = ('delivery_days', 'mean'),
).reset_index().round(2)
save(cat_perf, 'agg_category_performance.csv')
display(cat_perf.nlargest(10, 'total_revenue'))

  Saved agg_category_performance.csv: 498 rows x 9 cols


,product_category_name_english,order_year,order_quarter,total_revenue,order_count,unique_customers,avg_price,avg_review,avg_delivery
309,health_beauty,2018,2,343152.77,2078,2078,125.68,4.30,10.44
496,watches_gifts,2018,2,330210.52,1491,1491,184.69,4.15,12.23
113,computers_accessories,2018,1,319214.13,1995,1995,109.41,3.73,16.27
458,sports_leisure,2018,1,291349.26,1791,1791,120.51,3.83,15.22
308,health_beauty,2018,1,281850.29,1763,1763,123.88,3.97,15.14
64,bed_bath_table,2018,2,269187.02,1849,1849,94.56,3.96,10.63
310,health_beauty,2018,3,262216.17,1464,1464,138.01,4.32,8.83
495,watches_gifts,2018,1,261071.90,1033,1033,210.32,3.80,17.07
63,bed_bath_table,2018,1,259817.75,1929,1929,87.98,3.69,15.41
494,watches_gifts,2017,4,257463.52,1039,1039,206.42,4.04,13.25


In [24]:
# --- agg_state_performance: revenue by customer state (with lat/lng for maps) ---
state_perf = master.groupby(['customer_state', 'order_year', 'order_month']).agg(
    total_revenue    = ('revenue', 'sum'),
    order_count      = ('order_id', 'nunique'),
    unique_customers = ('customer_id', 'nunique'),
    avg_delivery     = ('delivery_days', 'mean'),
    avg_review       = ('review_score', 'mean'),
    lat              = ('lat_median_customer', 'mean'),
    lng              = ('lng_median_customer', 'mean'),
).reset_index().round(4)
save(state_perf, 'agg_state_performance.csv')

  Saved agg_state_performance.csv: 555 rows x 10 cols


In [25]:
# --- agg_payment_analysis: payment method breakdown ---
pay_perf = master.groupby(['payment_type', 'order_year', 'order_month']).agg(
    total_revenue    = ('revenue', 'sum'),
    total_payment    = ('payment_value', 'sum'),
    order_count      = ('order_id', 'nunique'),
    avg_installments = ('payment_installments', 'mean'),
    avg_review       = ('review_score', 'mean'),
).reset_index().round(2)
save(pay_perf, 'agg_payment_analysis.csv')

  Saved agg_payment_analysis.csv: 85 rows x 8 cols


In [26]:
# --- agg_review_sentiment: review scores by category and month ---
rev_fact = master[master['review_score'].notna()].copy()
review_agg = rev_fact.groupby(['product_category_name_english', 'order_ym']).agg(
    avg_rating   = ('review_score', 'mean'),
    review_count = ('review_score', 'count'),
    pct_positive = ('review_score', lambda x: (x >= 4).mean() * 100),
    pct_neutral  = ('review_score', lambda x: (x == 3).mean() * 100),
    pct_negative = ('review_score', lambda x: (x <= 2).mean() * 100),
).reset_index().round(2)
save(review_agg, 'agg_review_sentiment.csv')

  Saved agg_review_sentiment.csv: 1,261 rows x 7 cols


In [27]:
# --- dim_customers_rfm: RFM segmentation ---
ref_date = master['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
print(f'Reference date: {ref_date.date()}')

rfm = master.groupby('customer_id').agg(
    last_order_date  = ('order_purchase_timestamp', 'max'),
    first_order_date = ('order_purchase_timestamp', 'min'),
    frequency        = ('order_id', 'nunique'),
    monetary         = ('payment_value', 'sum'),
    avg_order_value  = ('payment_value', 'mean'),
    total_revenue    = ('revenue', 'sum'),
    avg_review       = ('review_score', 'mean'),
    customer_state   = ('customer_state', 'first'),
    customer_city    = ('customer_city', 'first'),
).reset_index()

rfm['recency_days'] = (ref_date - rfm['last_order_date']).dt.days
rfm['tenure_days']  = (ref_date - rfm['first_order_date']).dt.days
rfm = rfm.round(2)

rfm['R_score']   = pd.qcut(rfm['recency_days'], q=5, labels=[5,4,3,2,1]).astype(int)
rfm['F_score']   = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['M_score']   = pd.qcut(rfm['monetary'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['RFM_score'] = rfm['R_score'] * 100 + rfm['F_score'] * 10 + rfm['M_score']

def rfm_segment(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    if r >= 4 and f >= 4:              return 'Champions'
    elif r >= 3 and f >= 3:            return 'Loyal Customers'
    elif r >= 4 and f <= 2:            return 'New Customers'
    elif r >= 3 and m >= 3:            return 'Potential Loyalists'
    elif r <= 2 and f >= 3:            return 'At Risk'
    elif r <= 2 and f <= 2:            return 'Lost'
    elif r == 3 and f <= 2:            return 'About To Sleep'
    else:                              return 'Need Attention'

rfm['rfm_segment'] = rfm.apply(rfm_segment, axis=1)
save(rfm, 'dim_customers_rfm.csv')
display(rfm['rfm_segment'].value_counts().rename('count').to_frame())

Reference date: 2018-08-30
  Saved dim_customers_rfm.csv: 96,476 rows x 17 cols


,count
rfm_segment,
At Risk,23180
Loyal Customers,19361
New Customers,15551
Champions,15344
Lost,15267
Potential Loyalists,4593
About To Sleep,3180


In [28]:
# --- agg_cohort_retention: cohort retention matrix ---
first_p = master.groupby('customer_id')['order_purchase_timestamp'].min().reset_index()
first_p.columns = ['customer_id', 'cohort_date']
first_p['cohort_month'] = first_p['cohort_date'].dt.to_period('M')

cdf = master[['order_id', 'customer_id', 'order_purchase_timestamp']].copy()
cdf['order_month'] = cdf['order_purchase_timestamp'].dt.to_period('M')
cdf = cdf.merge(first_p[['customer_id', 'cohort_month']], on='customer_id')
cdf['period_number'] = (
    (cdf['order_month'].dt.year  - cdf['cohort_month'].dt.year)  * 12 +
    (cdf['order_month'].dt.month - cdf['cohort_month'].dt.month)
)

cohort_sizes = cdf.groupby('cohort_month')['customer_id'].nunique().reset_index()
cohort_sizes.columns = ['cohort_month', 'cohort_size']

retention = cdf.groupby(['cohort_month', 'period_number'])['customer_id'].nunique().reset_index()
retention.columns = ['cohort_month', 'period_number', 'active_customers']
retention = retention.merge(cohort_sizes, on='cohort_month')
retention['retention_rate'] = (retention['active_customers'] / retention['cohort_size'] * 100).round(2)
retention['cohort_month']   = retention['cohort_month'].astype(str)
retention = retention[retention['period_number'] <= 18]
save(retention, 'agg_cohort_retention.csv')

  Saved agg_cohort_retention.csv: 22 rows x 5 cols


In [29]:
# --- agg_seller_performance: seller-level metrics by month ---
seller_perf = master.groupby(['seller_id', 'seller_state', 'order_ym']).agg(
    total_revenue   = ('revenue', 'sum'),
    order_count     = ('order_id', 'nunique'),
    unique_products = ('product_id', 'nunique'),
    avg_price       = ('price', 'mean'),
    avg_delivery    = ('delivery_days', 'mean'),
    avg_review      = ('review_score', 'mean'),
).reset_index().round(2)
save(seller_perf, 'agg_seller_performance.csv')

  Saved agg_seller_performance.csv: 16,066 rows x 9 cols


In [30]:
# --- Final summary of all output files ---
import glob
files = sorted(glob.glob(os.path.join(OUT_DIR, '*.csv')))
total_mb = 0
print(f"{'File':<45} {'Rows':>8} {'MB':>6}")
print('-' * 62)
for f in files:
    nrows = sum(1 for _ in open(f, encoding='utf-8-sig')) - 1
    mb = os.path.getsize(f) / 1024 / 1024
    total_mb += mb
    print(f"{os.path.basename(f):<45} {nrows:>8,} {mb:>6.1f}")
print('-' * 62)
print(f"{'TOTAL':<45} {'':>8} {total_mb:>6.1f} MB")
print(f"\nOutput: {os.path.abspath(OUT_DIR)}")

File                                              Rows     MB
--------------------------------------------------------------
agg_category_performance.csv                       498    0.0
agg_cohort_retention.csv                            22    0.0
agg_monthly_kpi.csv                                 22    0.0
agg_payment_analysis.csv                            85    0.0
agg_review_sentiment.csv                         1,261    0.1
agg_seller_performance.csv                      16,066    1.1
agg_state_performance.csv                          555    0.0
dim_customers_rfm.csv                           96,476   11.4
fact_orders_enriched.csv                       115,719   48.6
--------------------------------------------------------------
TOTAL                                                    61.2 MB

Output: D:\ILPT\Project\AIO-Conquer\Project-1\data\tableau_data
